In [2]:
# Celda 1 - Conexión e imports para agentes municipales
from dao.mongo_dao import MongoDAO
from dao.agente_municipal_dao import AgenteMunicipalDAO
from bson import ObjectId

mongo = MongoDAO()
db = mongo.connect()
agente_dao = AgenteMunicipalDAO(mongo)
print("Conectado a DB:", mongo.db.name)


ImportError: cannot import name 'AgenteMunicipalDAO' from 'dao.agente_municipal_dao' (/home/dionel/Escritorio/Proyecto/CivicTech/dao/agente_municipal_dao.py)

In [2]:
# Celda A - Probar creación rápida de agente (usa un municipio real)
from bson import ObjectId

# Reemplazá por un municipio existente
municipio_id = "6a21d09b4076ce51229df8a3"

nuevo_agente = {
    "nombre_apellido": "Prueba Agente",
    "dni": "99999999",
    "email": "prueba.agente@municipio.local",
    "telefono": "+54 9 11 0000 0000",
    "cargo": "Prueba",
    "municipio_id": municipio_id,
    "activo": True
}

try:
    aid = agente_dao.crear(nuevo_agente)
    # Mostrar nombre del municipio
    try:
        mid = ObjectId(municipio_id)
        muni = mongo.db.municipio.find_one({"_id": mid}, {"nombre":1})
        muni_nombre = muni.get("nombre") if muni else "(no encontrado)"
    except Exception:
        muni_nombre = "(no disponible)"
    print("✅ Agente creado con _id:", aid)
    print("Pertenece al municipio:", muni_nombre)
except ValueError as e:
    print("⚠️ No se creó el agente:", e)
except Exception as e:
    import traceback
    print("❌ Error inesperado:", type(e).__name__, "-", e)
    traceback.print_exc()


⚠️ No se creó el agente: Error de validación en la colección: {'index': 0, 'code': 121, 'errmsg': 'Document failed validation'}


In [6]:
# Celda 1 - Mostrar validator de la colección agente_municipal
import json
info = mongo.db.command("listCollections", filter={"name":"agente_municipal"})["cursor"]["firstBatch"][0].get("options", {})
print("Opciones de la colección (options):")
print(json.dumps(info, default=str, indent=2))
print("\nValidator (si existe):")
print(json.dumps(info.get("validator"), default=str, indent=2))


Opciones de la colección (options):
{
  "validator": {
    "$jsonSchema": {
      "bsonType": "object",
      "required": [
        "nombre_apellido",
        "dni",
        "email",
        "rol",
        "municipio_id"
      ],
      "properties": {
        "nombre_apellido": {
          "bsonType": "string"
        },
        "dni": {
          "bsonType": "string"
        },
        "email": {
          "bsonType": "string"
        },
        "rol": {
          "bsonType": "string"
        },
        "municipio_id": {
          "bsonType": "objectId"
        },
        "acciones": {
          "bsonType": "array"
        }
      }
    }
  }
}

Validator (si existe):
{
  "$jsonSchema": {
    "bsonType": "object",
    "required": [
      "nombre_apellido",
      "dni",
      "email",
      "rol",
      "municipio_id"
    ],
    "properties": {
      "nombre_apellido": {
        "bsonType": "string"
      },
      "dni": {
        "bsonType": "string"
      },
      "email": {
        

In [3]:
# Celda - Insertar agente_municipal de prueba que cumple el validator
from bson import ObjectId
import datetime, traceback

municipio_id = "6a21d09b4076ce51229df8a3"

doc_ok = {
    "nombre_apellido": "Prueba Agente Municipal",
    "dni": "88888888",
    "email": "agente.prueba@municipio.local",
    "rol": "administrativo",            # campo requerido por el validator
    "municipio_id": None,               # se convertirá a ObjectId abajo
    "cargo": "Coordinador",             # campo de negocio opcional en DAO
    "acciones": [],                     # si el validator acepta array
    "telefono": "+54 9 11 0000 0000",
    "activo": True,
    "created_at": datetime.datetime.utcnow()
}

# Coerción segura de municipio_id a ObjectId
try:
    doc_ok["municipio_id"] = ObjectId(municipio_id)
except Exception:
    doc_ok["municipio_id"] = municipio_id  # si ya es ObjectId o falla, dejar como está

try:
    # Inserción directa para probar validación (evita DAO si querés ver error crudo)
    res = mongo.db.agente_municipal.insert_one(doc_ok)
    print("✅ Insertado directamente en agente_municipal con _id:", res.inserted_id)
except Exception as e:
    print("❌ Falló la inserción directa:", type(e).__name__, "-", e)
    # Si es WriteError, mostrar detalles si están disponibles
    details = getattr(e, "details", None)
    if details:
        import json
        print("Detalles del error:")
        print(json.dumps(details, default=str, indent=2))
    else:
        import traceback
        traceback.print_exc()


✅ Insertado directamente en agente_municipal con _id: 6a21ef2d5669d2ea4c5f3d5c


/tmp/ipykernel_74968/1328234046.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.datetime.utcnow()


In [2]:
# Crear agente municipal con todos los campos requeridos
from bson import ObjectId
import datetime

municipio_id = "<REEMPLAZAR_POR_UN_ID_DE_MUNICIPIO>"

nuevo_agente = {
    "nombre_apellido": "Agente Municipal DAO",
    "dni": "12345678",
    "email": "agente.municipal@municipio.local",
    "rol": "operativo",                      # requerido
    "municipio_id": ObjectId(municipio_id),  # debe ser ObjectId
    "cargo": "Inspector",
    "telefono": "+54 9 11 1234 5678",
    "activo": True,
    "acciones": [],                          # array permitido
    "created_at": datetime.datetime.now(datetime.timezone.utc)
}

try:
    aid = agente_dao.crear(nuevo_agente)
    print("✅ Agente creado con _id:", aid)
except Exception as e:
    print("❌ Falló la creación:", type(e).__name__, "-", e)
    details = getattr(e, "details", None)
    if details:
        import json
        print(json.dumps(details, default=str, indent=2))



❌ Falló la creación vía DAO: ValueError - Error de validación en la colección: {'index': 0, 'code': 121, 'errmsg': 'Document failed validation'}
